# 02. Data Cleaning & Exploratory Data Analysis (EDA)

This notebook performs data cleaning, quality assurance checks, and EDA on the **National Institute of Solar Energy (NISE)** inverter telemetry dataset.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(str(Path("..").resolve()))

from src.data_loader import load_nise_raw, extract_key_metrics
from src.preprocessing import clean_nise_data, save_processed_data

sns.set_theme(style="whitegrid")

## 1. Load Raw NISE Telemetry Data

In [ ]:
df_raw = load_nise_raw()
print("Raw Data Shape:", df_raw.shape)
df_metrics = extract_key_metrics(df_raw)
print("Metrics Data Shape:", df_metrics.shape)

## 2. Apply Data Quality & Physical Constraints

In [ ]:
df_clean = clean_nise_data(df_metrics)
print("Cleaned Data Shape:", df_clean.shape)
print("Missing Values Summary:")
print(df_clean[['timestamp', 'irradiance', 'total_ac_power_kw']].isnull().sum())
print()
print("Key Statistical Summary:")
print(df_clean[['irradiance', 'total_ac_power_kw']].describe())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Total AC Power Generation over Time

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_clean['timestamp'], df_clean['total_ac_power_kw'], color='#f39c12', linewidth=1.5, label='Total AC Power (kW)')
plt.xlabel('Timestamp')
plt.ylabel('Power (kW)')
plt.title('NISE Solar Plant Total AC Power Generation Over Time')
plt.legend()
plt.tight_layout()
plt.show()

### 3.2 Irradiance vs. Total AC Power Generation Scatter Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df_clean['irradiance'], df_clean['total_ac_power_kw'], alpha=0.4, color='#2980b9', edgecolors='none')
plt.xlabel('Solar Irradiance (RADIATION 500 KW)')
plt.ylabel('Total AC Power (kW)')
plt.title('Solar Irradiance vs PV Power Generation')
plt.tight_layout()
plt.show()

### 3.3 Average Diurnal (Hourly) Generation Profile

In [ ]:
hourly_power = df_clean.groupby('hour')['total_ac_power_kw'].mean()

plt.figure(figsize=(10, 5))
plt.bar(hourly_power.index, hourly_power.values, color='#27ae60', alpha=0.85)
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Mean Total AC Power (kW)')
plt.title('NISE Diurnal Solar Power Output Curve')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 4. Export Cleaned Dataset

In [ ]:
save_processed_data(df_clean)